In [16]:
from pathlib import Path
import pandas as pd


pr_root = Path.cwd().parent

# 2. $CO_{2}$ By Sector (WIP)

## 2.1 Dataset Overview and Cleaning

In [17]:
df = pd.read_csv(pr_root / "data" / "raw" / "co-emissions-by-sector" / "co-emissions-by-sector.csv")

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 6967 entries, 0 to 6966
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Entity                          6967 non-null   str    
 1   Code                            6967 non-null   str    
 2   Year                            6967 non-null   int64  
 3   Buildings                       6718 non-null   float64
 4   Industry                        6747 non-null   float64
 5   Land-use change and forestry    6967 non-null   int64  
 6   Other fuel combustion           6196 non-null   float64
 7   Transport                       6884 non-null   float64
 8   Manufacturing and construction  6441 non-null   float64
 9   Fugitive emissions              2449 non-null   float64
 10  Electricity and heat            6852 non-null   float64
 11  Aviation and shipping           6876 non-null   float64
dtypes: float64(8), int64(2), str(2)
memory usage:

,Entity,Code,Year,Buildings,Industry,Land-use change and forestry,Other fuel combustion,Transport,Manufacturing and construction,Fugitive emissions,Electricity and heat,Aviation and shipping
0,Afghanistan,AFG,1990,150000.0,50000.0,0,0.0,970000.0,570000.0,NaN,320000.0,20000.0
1,Afghanistan,AFG,1991,160000.0,50000.0,0,0.0,930000.0,530000.0,NaN,300000.0,20000.0
2,Afghanistan,AFG,1992,180000.0,50000.0,0,0.0,740000.0,390000.0,NaN,200000.0,20000.0
3,Afghanistan,AFG,1993,190000.0,50000.0,0,0.0,740000.0,380000.0,NaN,200000.0,20000.0
4,Afghanistan,AFG,1994,200000.0,50000.0,0,0.0,730000.0,360000.0,NaN,190000.0,20000.0


What we can gather:
- Besides information related to the year and the country, all the other columns are related to the sectors mentioned in 1.6.
- Each country has its own row for each year.
- We have some missing values in the dataset, let's find out how many and then document it.


In [18]:
df[df["Entity"] != "World"].isna().sum()



Entity                               0
Code                                 0
Year                                 0
Buildings                          249
Industry                           220
Land-use change and forestry         0
Other fuel combustion              771
Transport                           83
Manufacturing and construction     526
Fugitive emissions                4518
Electricity and heat               115
Aviation and shipping               91
dtype: int64

In [19]:
df_missing_values = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isna().sum(),
    'Missing_Percentage': (df.isna().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})
df_missing_values.to_csv(pr_root / "data" / "processed" / "missing_data_summary.csv", index=False)
df_missing_values

,Column,Missing_Count,Missing_Percentage,Data_Type
Entity,Entity,0,0.00,str
Code,Code,0,0.00,str
Year,Year,0,0.00,int64
Buildings,Buildings,249,3.57,float64
Industry,Industry,220,3.16,float64
Land-use change and forestry,Land-use change and forestry,0,0.00,int64
Other fuel combustion,Other fuel combustion,771,11.07,float64
Transport,Transport,83,1.19,float64
Manufacturing and construction,Manufacturing and construction,526,7.55,float64
Fugitive emissions,Fugitive emissions,4518,64.85,float64


In [ ]:
# Identify complete years (no missing values across sector columns)
sector_columns = [col for col in df.columns if col not in ['Entity', 'Code', 'Year']]

complete_years = df[df[sector_columns].notna().all(axis=1)]['Year'].unique()
incomplete_years = df[~df[sector_columns].notna().all(axis=1)]['Year'].unique()

print(f"Complete years: {sorted(complete_years)}")
print(f"Incomplete years: {sorted(incomplete_years)}")

# Store this info as a comment or in a separate reference file
missing_docs = {
    'complete_years': sorted(complete_years),
    'incomplete_years': sorted(incomplete_years),
    'sectors': sector_columns
}

In [ ]:
id_cols = ["Entity", "Code", "Year"]
sector_cols = [c for c in df.columns if c not in id_cols]

# One audit dataframe: one row per (Year, Sector)
missing_audit = (
    df.melt(
        id_vars=["Year"],
        value_vars=sector_cols,
        var_name="Sector",
        value_name="Value"
    )
    .assign(Is_Missing=lambda d: d["Value"].isna().astype(int))
    .drop(columns="Value")
)

# Missing values in each year
missing_per_year = (
    missing_audit.groupby("Year", as_index=False)["Is_Missing"].sum().rename(columns={"Is_Missing": "Missing_Count_Year"}))

# Optional: missing values by year and sector
missing_per_year_sector = (
    missing_audit.groupby(["Year", "Sector"], as_index=False)["Is_Missing"].sum().rename(columns={"Is_Missing": "Missing_Count"}))

# Save once, reuse later
missing_audit.to_csv(pr_root / "data" / "processed" / "missing_audit_long.csv", index=False)
missing_per_year.to_csv(pr_root / "data" / "processed" / "missing_per_year.csv", index=False)

# Worldwide data is also stored in this dataset. Let's override our dataframe to only have that.
df = df[df["Entity"] == "World"].reset_index(drop=True)

df["Buildings"]


Worldwide data is also stored in this dataset. Let's make override our dataframe to only have that.

In [20]:
df[df["Entity"] == "World"].head()

,Entity,Code,Year,Buildings,Industry,Land-use change and forestry,Other fuel combustion,Transport,Manufacturing and construction,Fugitive emissions,Electricity and heat,Aviation and shipping
6831,World,OWID_WRL,1990,2.611930e+09,492140000.0,2043360000,751820032.0,4.617160e+09,3.943070e+09,260010016.0,8.616370e+09,629340032.0
6832,World,OWID_WRL,1991,2.638740e+09,504149984.0,2043360000,752440000.0,4.654860e+09,3.867310e+09,276190016.0,8.758691e+09,637020032.0
6833,World,OWID_WRL,1992,2.531810e+09,525349984.0,2043360000,669270016.0,4.760600e+09,3.735450e+09,265790016.0,8.914870e+09,673960000.0
6834,World,OWID_WRL,1993,2.590980e+09,548820032.0,2043800064,684289984.0,4.802490e+09,3.681990e+09,271720000.0,8.965111e+09,666110016.0
6835,World,OWID_WRL,1994,2.510870e+09,586229952.0,2043469952,654619968.0,4.898900e+09,3.696790e+09,278460000.0,9.065770e+09,691859968.0


In [21]:
df = df[df["Entity"] == "World"].reset_index(drop=True)

In [22]:
df["Buildings"]

0     2.611930e+09
1     2.638740e+09
2     2.531810e+09
3     2.590980e+09
4     2.510870e+09
5     2.586560e+09
6     2.655770e+09
7     2.633550e+09
8     2.483140e+09
9     2.552570e+09
10    2.546160e+09
11    2.571720e+09
12    2.574850e+09
13    2.659820e+09
14    2.703530e+09
15    2.704730e+09
16    2.675380e+09
17    2.673850e+09
18    2.744840e+09
19    2.695770e+09
20    2.753160e+09
21    2.702110e+09
22    2.649000e+09
23    2.758280e+09
24    2.705700e+09
25    2.735200e+09
26    2.769370e+09
27    2.836290e+09
28    2.867690e+09
29    2.832650e+09
30    2.787250e+09
31    2.831580e+09
32    2.759500e+09
33    2.676930e+09
Name: Buildings, dtype: float64

## 2.2 Cleaning and Processing
## 2.2 Exploring the Dataset
## 2.3 Analysis of Trends

## 2.4 Conclusion